In [2]:
# 安装 pdf 处理工具、视觉检索库、交互 UI 库
!apt-get update && apt-get install -y poppler-utils
!pip install byaldi pdf2image gradio peft transformers datasets

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,452 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,893 kB] 
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18

In [3]:
import os
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import PeftModel
from byaldi import RAGMultiModalModel
import gradio as gr

# ==========================================
# 1. 路径定义 (严格匹配你的 Kaggle 数据结构)
# ==========================================
IMAGE_PATH = "/kaggle/input/datasets/somerchen/651655"
ADAPTER_PATH = "/kaggle/input/datasets/somerchen/123456/qwen35-2b-final"
BASE_MODEL_ID = "Qwen/Qwen3.5-2B"

# ==========================================
# 2. 初始化视觉检索大脑 (ColQwen2)
# ==========================================
print("🧠 正在初始化 ColQwen2 视觉检索引擎...")
retrieval_model = RAGMultiModalModel.from_pretrained("vidore/colqwen2-v1.0", device="cuda:0")

print("🔍 正在对论文截图进行视觉索引...")
retrieval_model.index(
    input_path=IMAGE_PATH,
    index_name="paper_db",
    store_collection_with_index=False,
    overwrite=True
)

# ==========================================
# 3. 加载生成大脑 (Base + LoRA)
# ==========================================
print("🤖 正在加载微调模型...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda:1"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# ==========================================
# 4. Agent 核心问答链路 (修复了路径报错)
# ==========================================
def paper_agent_qa(query):
    # 【Step 1: 视觉检索】
    results = retrieval_model.search(query, k=2)
    
    retrieved_images = []
    page_names = []
    
    for res in results:
        # --- 核心修复逻辑开始 ---
        # 尝试从 metadata 中获取原始文件名或路径
        # byaldi 的 Result 对象通常在 metadata 中存储 'doc_id' (文件名)
        raw_id = res.metadata.get('doc_id') or res.doc_id
        
        if isinstance(raw_id, int):
            # 如果还是整数，说明 metadata 里没存文件名，这通常不应该发生
            # 报错提示我们需要手动映射，但通常 raw_id 应该是文件名字符串
            print(f"警告: 检索到的 ID 为整数 {raw_id}，尝试查找文件...")
            # 备选方案：获取文件夹内文件列表按索引取（不推荐，除非万不得已）
            all_files = sorted([f for f in os.listdir(IMAGE_PATH) if f.endswith(('.png', '.jpg'))])
            img_filename = all_files[raw_id] if raw_id < len(all_files) else ""
        else:
            img_filename = str(raw_id)

        # 拼接成绝对路径
        if not os.path.isabs(img_filename):
            full_img_path = os.path.join(IMAGE_PATH, os.path.basename(img_filename))
        else:
            full_img_path = img_filename
        # --- 核心修复逻辑结束 ---

        try:
            img = Image.open(full_img_path).convert("RGB")
            retrieved_images.append(img)
            page_names.append(os.path.basename(full_img_path))
        except Exception as e:
            print(f"无法打开图片 {full_img_path}: {e}")

    if not retrieved_images:
        return "未能检索到相关页面或图片无法读取。", [], ""

    # 【Step 2: 理解生成】
    prompt = f"你是一个专业的论文助教，请结合提供的图片准确回答。问题：{query}"
    content = [{"type": "image"} for _ in retrieved_images] + [{"type": "text", "text": prompt}]
    messages = [{"role": "user", "content": content}]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = processor(
        text=[text], 
        images=retrieved_images, 
        return_tensors="pt",
        max_pixels=320*28*28 
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=300)
        output_ids = generated_ids[0][inputs.input_ids.shape[1]:]
        answer = processor.decode(output_ids, skip_special_tokens=True)
    
    return answer, retrieved_images, f"参考页面: {', '.join(page_names)}"

# ==========================================
# 5. 构建 Gradio 界面 (保持不变)
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📑 论文视觉阅读智能体 (修复版)")
    with gr.Row():
        with gr.Column(scale=1):
            query_box = gr.Textbox(label="输入问题")
            ask_btn = gr.Button("🧠 思考回答", variant="primary")
            ref_info = gr.Markdown()
        with gr.Column(scale=2):
            ans_box = gr.Textbox(label="Agent 回答", lines=10)
            img_gallery = gr.Gallery(label="参考页面截图", columns=2)
            
    ask_btn.click(fn=paper_agent_qa, inputs=query_box, outputs=[ans_box, img_gallery, ref_info])

demo.launch(share=True, inline=True)

🧠 正在初始化 ColQwen2 视觉检索引擎...
Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


adapter_config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

... (24 outputs left out)

✅ 模型挂载完成！


/tmp/ipykernel_55/2097166370.py:97: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b62b3d1132d23e18be.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 3519, in open
    fp.seek(0)
    ^^^^^^^
AttributeError: 'int' object has no attribute 'seek'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_t